<a href="https://colab.research.google.com/github/jacielefreitas63-tech/projeto-delivery-logistic/blob/main/1_limpeza_e_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Projeto Delivery: Engenharia de Dados & Análise Exploratória (EDA)


 # 1 Configuração do Ambiente e Conexão com o Data Lake (Google Drive)

Neste notebook, realizaremos a ingestão, limpeza fina, tratamento de outliers e união das bases transacionais da *Olist, malha rodoviária do **OpenStreetMap* e dados meteorológicos do *INMET*.

In [1]:
!pip install osmnx networkx pandas numpy pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 4.5 MB/s eta 0:00:00


In [2]:
# Importação das bibliotecas fundamentais para manipulação e análise estatística
import pandas as pd
import numpy as np
import networkx as nx
import osmnx as ox

# Imports específicos do PySpark para processamento distribuído
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
print("🚀 Bibliotecas carregadas com sucesso!")

🚀 Bibliotecas carregadas com sucesso!


In [3]:
# Inicialização do motor do Spark
spark = SparkSession.builder \
    .appName("LogisticaDelivery") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("🚀 Spark inicializado com sucesso e pronto para processamento distribuído!")

🚀 Spark inicializado com sucesso e pronto para processamento distribuído!


In [4]:
# Conexão segura com o armazenamento de dados do Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 📦2.Ingestão da Base de Pedidos (Olist Orders)
> O objetivo desta etapa é ler o dataset transacional de ordens de serviço, inspecionar os tipos de dados primitivos e mapear o volume inicial de registros antes de aplicar as regras de limpeza.

In [5]:
import pandas as pd
import os

# Caminho correto baseado na estrutura das suas imagens
base_path = '/content/drive/MyDrive/projeto_delivery_logistica/data/'

# Mapeamento dos arquivos que você listou
arquivos = {
    'customers': 'olist_customers_dataset.csv',
    'items': 'olist_order_items_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv'
}

# Dicionário para armazenar os DataFrames
df_raw = {}

# Carregamento automatizado
for nome, arquivo in arquivos.items():
    caminho_completo = os.path.join(base_path, arquivo)

    if os.path.exists(caminho_completo):
        df_raw[nome] = pd.read_csv(caminho_completo)
        print(f"✅ Arquivo '{arquivo}' carregado! Dimensões: {df_raw[nome].shape}")
    else:
        print(f"❌ Erro: O arquivo '{arquivo}' não foi encontrado em '{base_path}'")

# Verificação final dos arquivos carregados
print(f"\nTotal de bases carregadas: {len(df_raw)}")

✅ Arquivo 'olist_customers_dataset.csv' carregado! Dimensões: (99441, 5)
✅ Arquivo 'olist_order_items_dataset.csv' carregado! Dimensões: (112650, 7)
✅ Arquivo 'olist_orders_dataset.csv' carregado! Dimensões: (99441, 8)
✅ Arquivo 'olist_geolocation_dataset.csv' carregado! Dimensões: (1000163, 5)

Total de bases carregadas: 4


In [6]:
# Verifica quantos valores faltam em cada coluna
df_orders = df_raw['orders'] # FIX: Assign df_orders from df_raw dictionary
print(df_orders.isnull().sum())

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [7]:
# CÉLULA DE VALIDAÇÃO: Carregar e visualizar uma base tratada
nome_base = 'orders' # Você pode mudar para 'customers', 'items', etc.
caminho_parquet = f'/content/drive/MyDrive/projeto_delivery_logistica/data/{nome_base}_cleaned.parquet'

# Leitura do arquivo salvo
df_verificacao = pd.read_parquet(caminho_parquet)

# Exibir os dados e os tipos das colunas
print(f"--- Visualizando base: {nome_base} ---")
display(df_verificacao.head())
df_verificacao.info()

--- Visualizando base: orders ---


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [8]:
# Carregando as bases que foram salvas anteriormente
df_items = pd.read_parquet(os.path.join(base_path, 'items_cleaned.parquet'))
df_customers = pd.read_parquet(os.path.join(base_path, 'customers_cleaned.parquet'))
df_orders = pd.read_parquet(os.path.join(base_path, 'orders_cleaned.parquet'))

# Agora você pode rodar o merge
df_completo = pd.merge(df_orders, df_items, on='order_id', how='left')
df_completo = pd.merge(df_completo, df_customers, on='customer_id', how='left')

 Consolidação (Merge) das Bases

In [9]:
# Unindo Pedidos com Itens
df_completo = pd.merge(df_orders, df_items, on='order_id', how='left')

# Agora unindo o resultado com Clientes
df_completo = pd.merge(df_completo, df_customers, on='customer_id', how='left')

 Conversão de Tipos (Cast) e Tratamento de Datas
> As colunas temporais de carimbo de data/hora (timestamps) precisam ser convertidas de object para datetime64. Isso nos permitirá calcular intervalos de tempo, como o tempo de entrega real (SLA) e atrasos.

In [10]:

# Carrega a base 'orders' que você já tratou e salvou em parquet
base_path = '/content/drive/MyDrive/projeto_delivery_logistica/data/'
df_orders = pd.read_parquet(os.path.join(base_path, 'orders_cleaned.parquet'))

# Descobre o intervalo de tempo dos seus pedidos
data_min = df_orders['order_purchase_timestamp'].min()
data_max = df_orders['order_purchase_timestamp'].max()

print(f"✅ O seu projeto abrange o período de: {data_min.date()} até {data_max.date()}")

✅ O seu projeto abrange o período de: 2016-09-04 até 2018-10-17


In [11]:
import pandas as pd
import os

# --- Configuração de Caminhos ---
# Define onde os arquivos de origem (Bronze) estão localizados
BASE_PATH = '/content/drive/MyDrive/projeto_delivery_logistica/data/'

# --- Configuração de Regras de Negócio (Tipagem) ---
# Mapeamento para conversão de colunas de data em cada dataset.
# Esta estrutura permite fácil manutenção caso novas tabelas sejam adicionadas.
SCHEMA_CONFIG = {
    'orders': {
        'cols_to_date': ['order_purchase_timestamp', 'order_approved_at',
                         'order_delivered_carrier_date', 'order_delivered_customer_date',
                         'order_estimated_delivery_date']
    },
    'items': {
        'cols_to_date': ['shipping_limit_date']
    },
    'customers': {'cols_to_date': []}, # Sem datas para tratar
    'products': {'cols_to_date': []}   # Sem datas para tratar
}

# Lista dos arquivos brutos (origem)
arquivos = {
    'customers': 'olist_customers_dataset.csv',
    'items': 'olist_order_items_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv'
}

# --- Execução do Pipeline (ETL) ---
for nome, arquivo in arquivos.items():
    caminho_entrada = os.path.join(BASE_PATH, arquivo)

    # 1. EXTRAÇÃO (Extract): Carrega o dataset original
    df = pd.read_csv(caminho_entrada)

    # 2. TRANSFORMAÇÃO (Transform): Limpeza e normalização
    # Aplica conversão para datetime apenas nas colunas definidas no SCHEMA_CONFIG
    if nome in SCHEMA_CONFIG:
        for col in SCHEMA_CONFIG[nome]['cols_to_date']:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')

    # 3. CARGA (Load): Salva o dataset tratado individualmente
    # Utiliza o formato Parquet para maior eficiência de I/O e compressão de dados
    caminho_saida = os.path.join(BASE_PATH, f'{nome}_cleaned.parquet')
    df.to_parquet(caminho_saida, index=False)

    print(f"✅ [SUCCESS] {nome.upper()} processada e salva em: {caminho_saida}")

print("\n🚀 Todos os datasets foram tratados e persistidos com sucesso.")

✅ [SUCCESS] CUSTOMERS processada e salva em: /content/drive/MyDrive/projeto_delivery_logistica/data/customers_cleaned.parquet
✅ [SUCCESS] ITEMS processada e salva em: /content/drive/MyDrive/projeto_delivery_logistica/data/items_cleaned.parquet
✅ [SUCCESS] ORDERS processada e salva em: /content/drive/MyDrive/projeto_delivery_logistica/data/orders_cleaned.parquet
✅ [SUCCESS] GEOLOCATION processada e salva em: /content/drive/MyDrive/projeto_delivery_logistica/data/geolocation_cleaned.parquet

🚀 Todos os datasets foram tratados e persistidos com sucesso.


In [12]:
# Lista de colunas que contêm datas no dataset
colunas_datas = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# Converter todas as colunas da lista para o formato datetime
for coluna in colunas_datas:
    df_orders[coluna] = pd.to_datetime(df_orders[coluna])

# Validar se a conversão funcionou checando os novos tipos
df_orders[colunas_datas].dtypes

,0
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


 Análise de Valores Ausentes (Missing Values)
> Antes de qualquer modelagem preditiva, mapeamos o volume absoluto e percentual de nulos para decidir entre estratégias de imputação ou remoção de registros.

In [13]:
# Calcular o total de nulos por coluna
total_nulos = df_orders.isnull().sum()

# Calcular a porcentagem de nulos por coluna
porcentagem_nulos = (df_orders.isnull().sum() / len(df_orders)) * 100

# Juntar as informações em um DataFrame de diagnóstico para o relatório
diagnostico_nulos = pd.DataFrame({'Total Nulos': total_nulos, 'Porcentagem (%)': porcentagem_nulos})
diagnostico_nulos.sort_values(by='Total Nulos', ascending=False)

,Total Nulos,Porcentagem (%)
order_delivered_customer_date,2965,2.981668
order_delivered_carrier_date,1783,1.793023
order_approved_at,160,0.160899
order_id,0,0.000000
order_purchase_timestamp,0,0.000000
order_status,0,0.000000
customer_id,0,0.000000
order_estimated_delivery_date,0,0.000000


## 🌤️ Ingestão e Tratamento de Dados Meteorológicos (INMET)
> O objetivo desta etapa é carregar o histórico climático, tratar valores nulos de precipitação (chuva) e preparar a base para cruzamento temporal com os momentos de compra e entrega.

In [14]:
from pyspark.sql import SparkSession

# Inicia a sessão
spark = SparkSession.builder.appName("CarregarDados").getOrCreate()

# Caminho completo e direto
BASE_PATH = '/content/drive/MyDrive/projeto_delivery_logistica/data/'


In [15]:
# Carregando os dados brutos do INMET
df_southeast = spark.read.csv(BASE_PATH + 'southeast.csv', header=True, inferSchema=True)
df_stations = spark.read.csv(BASE_PATH + 'stations.csv', header=True, inferSchema=True)

# Verificando a estrutura
print("--- Estrutura do DataFrame Southeast ---")
df_southeast.printSchema()
df_southeast.show(5)

--- Estrutura do DataFrame Southeast ---
root
 |-- index: integer (nullable = true)
 |-- Data: date (nullable = true)
 |-- Hora: timestamp (nullable = true)
 |-- PRECIPITAÇÃO TOTAL, HORÁRIO (mm): double (nullable = true)
 |-- PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB): double (nullable = true)
 |-- PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB): double (nullable = true)
 |-- PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB): double (nullable = true)
 |-- RADIACAO GLOBAL (Kj/m²): integer (nullable = true)
 |-- TEMPERATURA DO AR - BULBO SECO, HORARIA (°C): double (nullable = true)
 |-- TEMPERATURA DO PONTO DE ORVALHO (°C): double (nullable = true)
 |-- TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C): double (nullable = true)
 |-- TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C): double (nullable = true)
 |-- TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C): double (nullable = true)
 |-- TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C): double (nullable = true)
 |-- UMIDADE REL. MAX. NA H

 Correção do Clima (df_southeast) Este código cria o timestamp correto e trata os nulos de radiação e precipitação.

In [16]:
from pyspark.sql.functions import col, when, regexp_replace

# 1. Carrega o arquivo forçando tudo como STRING inicialmente
# Isso impede que o Spark tente adivinhar o tipo e trave no carregamento
df_southeast = spark.read.option("header", "true").csv(BASE_PATH + 'southeast.csv')

# 2. Lista de colunas que devem ser numéricas
cols_numericas = [
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)", # Corrigido: sem acento em PRESSAO
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)" # Corrigido: sem acento em HORARIA
]

# 3. Tratamento: Substitui vírgula por ponto (caso tenha) e remove espaços vazios
for col_name in cols_numericas:
    # Substitui ',' por '.' e converte para double
    df_southeast = df_southeast.withColumn(
        col_name,
        regexp_replace(col(col_name), ",", ".").cast("double")
    )

    # Substitui o valor -9999 por NULL (nulo)
    df_southeast = df_southeast.withColumn(
        col_name,
        when(col(col_name) == -9999, None).otherwise(col(col_name))
    )

print("✅ Limpeza concluída sem erros de formato!")

✅ Limpeza concluída sem erros de formato!


In [17]:
df_southeast.select(cols_numericas).summary().show()

+-------+--------------------------------+-----------------------------------------------------+--------------------------------------------+
|summary|PRECIPITAÇÃO TOTAL, HORÁRIO (mm)|PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)|TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)|
+-------+--------------------------------+-----------------------------------------------------+--------------------------------------------+
|  count|                        14020720|                                             14230752|                                    14313954|
|   mean|              0.1493912580808481|                                     948.421337178806|                          21.970964235318892|
| stddev|              1.2260827677652342|                                    43.27349823998682|                           5.345386048219098|
|    min|                             0.0|                                                720.1|                                       -51.4|
|    2

In [19]:
# Definindo o nome do arquivo final
caminho_final = BASE_PATH + "inmet_southeast_final.parquet"

# Salva em um arquivo só (coalesce(1)) para facilitar a leitura futura
df_southeast.coalesce(1).write.mode("overwrite").parquet(caminho_final)

print(f"✅ Base INMET (Southeast) tratada com sucesso e salva em: {caminho_final}")

✅ Base INMET (Southeast) tratada com sucesso e salva em: /content/drive/MyDrive/projeto_delivery_logistica/data/inmet_southeast_final.parquet


📂 Tratamento e Salvamento da Base stations
A base foi carregada e salva no formato Parquet, garantindo uma estrutura otimizada para o futuro join com os dados climáticos.

In [21]:
from pyspark.sql.functions import trim

# 1. Limpeza de espaços em branco (trim) para evitar erros de JOIN
df_stations_corrigido = df_stations
for col_name in df_stations.columns:
    if df_stations.schema[col_name].dataType.simpleString() == 'string':
        df_stations_corrigido = df_stations_corrigido.withColumn(col_name, trim(col(col_name)))


In [22]:
# 2. Salvar as estações tratadas
df_stations_corrigido.coalesce(1).write.mode("overwrite").parquet(BASE_PATH + "stations_final.parquet")
print("✅ Estações tratadas e salvas!")

✅ Estações tratadas e salvas!


In [25]:
# Carrega o arquivo das estações
df_estacoes = spark.read.parquet(BASE_PATH + "stations_final.parquet")

# Exibe os dados
df_estacoes.show(10)

+--------------------+------+-----+------------+----------+-------+------------+------------+
|             station|region|state|station_code|first date| height|   longitude|    latitude|
+--------------------+------+-----+------------+----------+-------+------------+------------+
|           QUEIMADAS|    NE|   BA|        A436|2008-05-23|  315.0|-39.61694443|  -10.984645|
|               MACAU|    NE|   RN|        A317|2007-01-06|   32.0|-36.57305554| -5.15111111|
|SAQUAREMA - SAMPA...|    SE|   RJ|        A667|2019-01-01|   26.0|-42.60888888| -22.8711111|
|SANTANA DO LIVRAM...|     S|   RS|        A804|2001-11-22|  328.0|-55.40138888|-30.75055555|
|          VILA VELHA|    SE|   ES|        A634|2017-02-15|   25.0|-40.40388888|-20.46694443|
|GOVERNADOR VALADARES|    SE|   MG|        A532|2007-05-27|  214.0|-41.97666666|-18.83027777|
|             CABROBO|    NE|   PE|        A329|2007-09-04| 342.74|-39.31388888| -8.50305555|
|         OURO BRANCO|    SE|   MG|        A513|2006-07-28| 

## 🗺️ 6. Ingestão e Tratamento de Dados de Malha Rodoviária (OpenStreetMap)
> Nesta etapa, mapeamos as coordenadas geográficas (latitude e longitude) dos clientes e vendedores para calcular as distâncias reais das rotas e entender o impacto do tráfego e das rodovias no tempo de entrega.

In [26]:
import osmnx as ox
import pandas as pd

# Definindo a área de interesse (Ex: Estado de São Paulo)
lugar = "São Paulo, Brazil"

# Baixando o grafo (network_type='drive' é o padrão para logística)
# NOTA: O OSMnx baixa em memória, por isso limitamos a área.
grafo = ox.graph_from_place(lugar, network_type='drive')

# Convertendo grafo para DataFrames (nodes e edges)
nodes_gdf, edges_gdf = ox.graph_to_gdfs(grafo)

# Adicionando coluna de partição para organização
edges_gdf['estado'] = 'SP'
print(f"✅ Malha rodoviária carregada: {len(edges_gdf)} trechos encontrados.")

✅ Malha rodoviária carregada: 304834 trechos encontrados.


Tratamento e Limpeza
Aqui focamos em preparar os dados para o cálculo de distâncias. Removemos colunas desnecessárias para reduzir o peso do arquivo.

In [29]:
# 1. Antes de converter para tabela, vamos processar o grafo original
print("Calculando velocidades e tempos no grafo...")
grafo = ox.add_edge_speeds(grafo)
grafo = ox.add_edge_travel_times(grafo)

# 2. Agora sim, convertemos para GeoDataFrame (tabela)
nodes_gdf, edges_gdf = ox.graph_to_gdfs(grafo)

# 3. Resetamos o índice para transformar 'u', 'v' e 'key' em colunas normais
edges_gdf = edges_gdf.reset_index()

# 4. Selecionamos apenas as colunas que realmente existem
# (Usamos um filtro para evitar KeyError caso alguma não exista)
colunas_desejadas = ['u', 'v', 'length', 'highway', 'speed_kph', 'travel_time']
colunas_finais = [c for c in colunas_desejadas if c in edges_gdf.columns]

edges_clean = edges_gdf[colunas_finais].copy()
edges_clean['estado'] = 'SP'

# 5. Tratamento de limpeza final
edges_clean['length'] = edges_clean['length'].astype(float)
if 'speed_kph' in edges_clean.columns:
    edges_clean['speed_kph'] = edges_clean['speed_kph'].fillna(50.0)

print(f"✅ Tratamento concluído! Colunas processadas: {edges_clean.columns.tolist()}")

Calculando velocidades e tempos no grafo...
✅ Tratamento concluído! Colunas processadas: ['u', 'v', 'length', 'highway', 'speed_kph', 'travel_time', 'estado']


Validação
Uma etapa crítica de controle de qualidade para garantir que a malha não possui dados corrompidos.

In [30]:
# Verificação de integridade
assert 'length' in edges_clean.columns, "Coluna de distância não encontrada!"
assert not edges_clean['length'].isnull().any(), "Existem distâncias nulas na malha!"

print(f"Dados validados. Total de registros: {len(edges_clean)}")
print(edges_clean.head())

Dados validados. Total de registros: 304834
        u           v      length    highway  speed_kph  travel_time estado
0  573641   465879071   25.402787      trunk  60.000000     1.524167     SP
1  573641  4345300990   33.384162   tertiary  41.434394     2.900561     SP
2  573643   292424978  286.802985      trunk  60.000000    17.208179     SP
3  573644    25619056  324.413125   motorway  75.000000    15.571830     SP
4  577239  6549930664  178.305444  secondary  90.000000     7.132218     SP


Salvamento com Particionamento
O uso do partitionBy organiza seu Data Lake em pastas, permitindo que o Spark leia apenas o que é necessário no futuro.

In [33]:
import numpy as np

# 1. Seleciona apenas colunas que possuem tipos numéricos ou string simples
# Isso elimina qualquer coluna com listas, dicionários ou objetos complexos
cols_to_keep = []
for col in edges_clean.columns:
    # Verifica se a coluna não é do tipo 'object' (onde ficam as listas)
    if edges_clean[col].dtype != 'object':
        cols_to_keep.append(col)
    else:
        # Se for object, tentamos converter para string (texto simples)
        edges_clean[col] = edges_clean[col].astype(str)
        cols_to_keep.append(col)

# Cria uma cópia apenas com as colunas limpas
df_final_spark = edges_clean[cols_to_keep].copy()

# 2. Conversão segura para Spark
spark_df = spark.createDataFrame(df_final_spark)

# 3. Salvamento particionado
caminho_particionado = BASE_PATH + "malha_rodoviaria_particionada/"

spark_df.write.mode("overwrite") \
    .partitionBy("estado") \
    .parquet(caminho_particionado)

print(f"✅ Dados salvos com sucesso em: {caminho_particionado}")

✅ Dados salvos com sucesso em: /content/drive/MyDrive/projeto_delivery_logistica/data/malha_rodoviaria_particionada/


In [36]:
# 1. Carrega os dados que você acabou de salvar particionados
df_malha_verificacao = spark.read.parquet(BASE_PATH + "malha_rodoviaria_particionada/")

# 2. Mostra as primeiras linhas
df_malha_verificacao.show(5)

# 3. Confere se o particionamento funcionou (deve aparecer a coluna 'estado')
df_malha_verificacao.printSchema()

+----------+----------+------------------+-----------+------------------+------------------+------+
|         u|         v|            length|    highway|         speed_kph|       travel_time|estado|
+----------+----------+------------------+-----------+------------------+------------------+------+
|1733441599|1728215446| 56.65440734619385|residential|35.431968584102975| 5.756266857207741|    SP|
|1733441599|1733441453|207.78194321645316|residential|35.431968584102975|21.111302179095922|    SP|
|1733441599|1733441595| 53.34779199246641|residential|35.431968584102975|  5.42030428586025|    SP|
|1733441604|1733441610|53.734690633187014|residential|35.431968584102975| 5.459614410650185|    SP|
|1733441604|1733441454| 209.8743656979118|residential|35.431968584102975|21.323898916851856|    SP|
+----------+----------+------------------+-----------+------------------+------------------+------+
only showing top 5 rows
root
 |-- u: long (nullable = true)
 |-- v: long (nullable = true)
 |-- leng

## 🚚 4. Engenharia de Atributos (Feature Engineering): Métricas de SLA
> Criaremos as variáveis preditivas e de performance logística:
> 1. **tempo_entrega_dias**: Tempo real gasto desde a compra até a chegada ao cliente.
> 2. **atraso_entrega_dias**: Dias de atraso caso a entrega passe do prazo estimado (valores positivos indicam quebra de SLA).

In [37]:
# 1. Calcular o tempo de entrega real em dias
df_orders['tempo_entrega_dias'] = (df_orders['order_delivered_customer_date'] - df_orders['order_purchase_timestamp']).dt.total_seconds() / 86400

# 2. Calcular o atraso em relação à estimativa (Valores > 0 significam atraso)
df_orders['atraso_entrega_dias'] = (df_orders['order_delivered_customer_date'] - df_orders['order_estimated_delivery_date']).dt.total_seconds() / 86400

# Substituir valores negativos por 0 no atraso (pois entregas antes do prazo não têm atraso)
df_orders['atraso_entrega_dias'] = df_orders['atraso_entrega_dias'].clip(lower=0)

# Exibir uma amostragem com as novas colunas calculadas
df_orders[['order_id', 'tempo_entrega_dias', 'atraso_entrega_dias']].head()

,order_id,tempo_entrega_dias,atraso_entrega_dias
0,e481f51cbdc54678b7cc49136f2d6af7,8.436574,0.0
1,53cdb2fc8bc7dce0b6741e2150273451,13.782037,0.0
2,47770eb9100c2d0c44946d9cf07ec65d,9.394213,0.0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.208750,0.0
4,ad21c59c0840e6cb83a9ceb5573f8159,2.873877,0.0


In [38]:
# Lista todas as variáveis que existem na memória agora
%whos

Variable                Type            Data/Info
-------------------------------------------------
BASE_PATH               str             /content/drive/MyDrive/pr<...>_delivery_logistica/data/
F                       module          <module 'pyspark.sql.func<...>l/functions/__init__.py'>
SCHEMA_CONFIG           dict            n=4
SparkSession            type            <class 'pyspark.sql.session.SparkSession'>
arquivo                 str             olist_geolocation_dataset.csv
arquivos                dict            n=4
base_path               str             /content/drive/MyDrive/pr<...>_delivery_logistica/data/
caminho_completo        str             /content/drive/MyDrive/pr<...>t_geolocation_dataset.csv
caminho_entrada         str             /content/drive/MyDrive/pr<...>t_geolocation_dataset.csv
caminho_final           str             /content/drive/MyDrive/pr<...>t_southeast_final.parquet
caminho_parquet         str             /content/drive/MyDrive/pr<...>ta/orders_cle

#📊 7. Integração e Análise de Impacto Climático
Nesta etapa, utilizamos o PySpark para processar o histórico meteorológico de forma distribuída, garantindo eficiência no cruzamento de dados massivos. O objetivo é consolidar as informações climáticas (precipitação) com as métricas logísticas da Olist, permitindo a identificação de padrões onde eventos climáticos severos impactam diretamente o SLA de entrega.

In [39]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_date, avg, max

# Ensure Spark session is available in this cell's context
spark = SparkSession.builder \
    .appName("LogisticaDelivery") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# --- PREPARATION: Define df_pedidos (from df_orders) and df_clima (from df_southeast) ---

# 1. Prepare df_pedidos from df_orders (Pandas to Spark)
# Ensure df_orders is available (it should be from previous cells)
# df_orders is a Pandas DataFrame, convert it to Spark
# and extract the date part for 'data_pedido'
df_pedidos = spark.createDataFrame(df_orders) \
                  .withColumnRenamed("order_purchase_timestamp", "data_completa_pedido") \
                  .withColumn("data_pedido", to_date(col("data_completa_pedido")))

# 2. Prepare df_clima from df_southeast (Spark DataFrame)
# Aggregate df_southeast by date to get daily average and max precipitation
# Rename 'Data' to 'data_pedido' and precipitation columns
df_clima_raw = df_southeast.withColumnRenamed("PRECIPITAÇÃO TOTAL, HORÁRIO (mm)", "precipitacao_total_horario_mm")
df_clima = df_clima_raw.groupBy("Data").agg(
    avg("precipitacao_total_horario_mm").alias("media_chuva_mm"),
    max("precipitacao_total_horario_mm").alias("max_chuva_mm")
).withColumnRenamed("Data", "data_pedido") # Rename Data to data_pedido for joining


# --- ORIGINAL CODE FROM THE CELL ---

# 1. Realiza o join (o cruzamento dos dados)
df_pedidos_clima = df_pedidos.join(df_clima, ["data_pedido"], "left")

# 2. Limpeza Profissional (o "pulo do gato")
# Removemos linhas essenciais que faltam dados e preenchemos o resto
df_limpo = df_pedidos_clima.dropna(subset=["atraso_entrega_dias", "tempo_entrega_dias"]) \
                           .fillna({"media_chuva_mm": 0, "max_chuva_mm": 0}) \
                           .withColumn("media_chuva_mm", when(col("media_chuva_mm") < 0, 0).otherwise(col("media_chuva_mm")))

# 3. Registra a View com o nome que usaremos nas consultas SQL
df_limpo.createOrReplaceTempView("pedidos_clima_limpo")

# 4. Exibe uma amostra para você confirmar que não há mais NaN
df_limpo.select("data_pedido", "atraso_entrega_dias", "media_chuva_mm").show(5)

KeyboardInterrupt: 

In [ ]:
# --- Célula 7: Agregação e Integração Otimizada ---
from pyspark.sql import functions as F

# Nomes exatos conforme aparecem no seu printSchema()
col_data = "Data"
col_chuva = "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)"

# 1. Agrupar no Spark (Processamento Distribuído)
df_clima_diario_spark = df_sp.groupBy(col_data).agg(
    F.avg(col_chuva).alias("media_chuva_mm"),
    F.max(col_chuva).alias("max_chuva_mm")
)

# 2. Trazer para o Pandas APENAS o resultado consolidado
df_clima_diario = df_clima_diario_spark.toPandas()

# 3. Preparar colunas para o merge
# Certifique-se de que os nomes das colunas de data sejam idênticos nos dois DataFrames
df_clima_diario['Data'] = pd.to_datetime(df_clima_diario['Data'])
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
df_orders['Data'] = df_orders['order_purchase_timestamp'].dt.normalize()

# 4. Realizar o Merge
df_final = pd.merge(df_orders, df_clima_diario, on='Data', how='left')
df_final['media_chuva_mm'] = df_final['media_chuva_mm'].fillna(0)

print("✅ Integração concluída com sucesso!")
display(df_final.head())

## 8. Identificação de Variáveis e Correlação
Análise estatística para validar se o aumento da pluviosidade (chuva) impacta o 'atraso_entrega_dias'.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Selecionar colunas numéricas para correlação
colunas_analise = ['media_chuva_mm', 'atraso_entrega_dias', 'tempo_entrega_dias']
correlacao = df_final[colunas_analise].corr()

# Plotar Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlacao, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlação: Chuva vs Performance Logística")
plt.show()

In [ ]:
# Análise de Correlação

# 1. Calcular a matriz de correlação
# Queremos ver a relação entre 'media_chuva_mm' e 'atraso_entrega_dias'
correlacao = df_final[['media_chuva_mm', 'atraso_entrega_dias']].corr()

# 2. Plotar o mapa de calor (Heatmap)
plt.figure(figsize=(8, 6))
sns.heatmap(correlacao, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlação: Chuva vs Atraso na Entrega")
plt.show()

# 3. Plotar um gráfico de dispersão para visualizar a tendência
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df_final, x='media_chuva_mm', y='atraso_entrega_dias', alpha=0.5)
plt.title("Dispersão: Intensidade da Chuva x Atraso")
plt.show()

#🔍 9. Análise Estruturada com Linguagem SQL
Nesta etapa, utilizamos a biblioteca pandasql para aplicar consultas SQL diretamente sobre o dataframe consolidado, extraindo métricas de negócio que validam nossas hipóteses

In [ ]:
# --- 🔍 9. Análise Estruturada com PySpark SQL ---

# 1. Converter o DataFrame do Pandas (df_final) de volta para o Spark
df_final_spark = spark.createDataFrame(df_final)

# 2. Registrar o DataFrame do Spark como uma View temporária
df_final_spark.createOrReplaceTempView("pedidos_clima")

# 3. Executar a consulta usando o motor nativo do Spark
query = """
SELECT
    CASE
        WHEN media_chuva_mm = 0 THEN 'Sem Chuva'
        WHEN media_chuva_mm BETWEEN 0.1 AND 10 THEN 'Chuva Leve'
        WHEN media_chuva_mm > 10 THEN 'Chuva Intensa'
        ELSE 'Indeterminado'
    END as categoria_chuva,
    COUNT(*) as total_pedidos,
    AVG(atraso_entrega_dias) as media_atraso_dias
FROM pedidos_clima
GROUP BY 1
ORDER BY media_atraso_dias DESC
"""

# 4. Executa a query e mostra o resultado
resultado = spark.sql(query)
resultado.show()

In [ ]:
# 1. Converter as colunas para tipo numérico e remover nulos
df_tratado = df_final_spark.withColumn("atraso_entrega_dias", df_final_spark["atraso_entrega_dias"].cast("double")) \
                           .withColumn("tempo_entrega_dias", df_final_spark["tempo_entrega_dias"].cast("double")) \
                           .na.fill(0, subset=["atraso_entrega_dias", "tempo_entrega_dias"])

# 2. Atualizar a View com os dados tratados
df_tratado.createOrReplaceTempView("pedidos_clima")

# 3. Agora rode a query de estatísticas novamente
query_stats = """
SELECT
    AVG(atraso_entrega_dias) as media_atraso,
    MAX(atraso_entrega_dias) as maior_atraso,
    AVG(tempo_entrega_dias) as media_tempo_entrega
FROM pedidos_clima
"""
spark.sql(query_stats).show()

In [ ]:
# Convertendo para Spark DataFrame e registrando a view
df_final_spark = spark.createDataFrame(df_final)
df_final_spark.createOrReplaceTempView("pedidos_clima")

# Query: Agrupa por mês para verificar a sazonalidade
query_sazonalidade = """
SELECT
    month(Data) as mes,            -- Extrai o mês da coluna de Data
    AVG(media_chuva_mm) as media_chuva_mes, -- Calcula a média de chuva no mês
    COUNT(order_id) as total_pedidos         -- Conta o total de pedidos realizados
FROM pedidos_clima
GROUP BY 1                         -- Agrupa pelo mês (primeira coluna)
ORDER BY 1                         -- Ordena de janeiro a dezembro
"""

spark.sql(query_sazonalidade).show()

In [ ]:
# Verifica as colunas disponíveis na View
spark.table("pedidos_clima").printSchema()

In [ ]:
# Analisa dias com chuva (media_chuva_mm > 0) e atraso (atraso_entrega_dias > 0)
query_chuva_atraso = """
SELECT
    Data,
    media_chuva_mm,
    atraso_entrega_dias
FROM pedidos_clima
WHERE media_chuva_mm > 0 AND atraso_entrega_dias > 0
ORDER BY atraso_entrega_dias DESC
LIMIT 10
"""
spark.sql(query_chuva_atraso).show()

In [ ]:
# Calcula a média e o tempo máximo de atraso encontrados na base
query_stats = """
SELECT
    AVG(atraso_entrega_dias) as media_atraso,
    MAX(atraso_entrega_dias) as maior_atraso,
    AVG(tempo_entrega_dias) as media_tempo_entrega
FROM pedidos_clima
"""
spark.sql(query_stats).show()